# Attention U-Net - Sentinel-2 land cover (built-up / water / trees / barren)

Classic U-Net + an attention gate on every skip connection, trained from scratch.

**Flow:** 1. look at the data with its colored labels  2. train  3. training curves + confusion matrices  4. test-set results

Kaggle settings needed: **GPU on**.

## 0. Settings

In [ ]:
# ===================== SETTINGS (edit here) =====================
MODEL      = "attn_unet"          # unet | attn_unet | resnet_unet | attn_resnet_unet
SEED       = 1                    # change to 2, 3 and re-run for the other seeds
EPOCHS     = 100
BATCH_SIZE = 16
DATA_ROOT  = "/kaggle/working/DATASET"   # folder containing Train/ Validation/ Test/ (auto-detected below if not there)
OUT_ROOT   = "/kaggle/working/runs"
RETRAIN    = False                # False = skip training if this run already finished
N_SHOW     = 6                    # tiles shown in the picture grids
# ================================================================

import glob, os
if not os.path.isdir(os.path.join(DATA_ROOT, "Train")):
    _found = glob.glob("/kaggle/input/**/Train", recursive=True)
    if _found:
        DATA_ROOT = os.path.dirname(_found[0])
print("DATA_ROOT =", DATA_ROOT)

In [ ]:
import os, sys, json, glob, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from IPython.display import display

# find the folder that holds train.py / model.py / dataset.py
_cand = ["/kaggle/input/datasets/janvisoni23/unet-models/unet_project", "/kaggle/working/unet_project", "/kaggle/working", os.getcwd()]
_cand += [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/train.py", recursive=True)]
_cand += [os.path.dirname(p) for p in glob.glob("/kaggle/working/**/train.py", recursive=True)]
PROJECT_DIR = next((d for d in _cand if os.path.exists(os.path.join(d, "train.py"))), None)
if PROJECT_DIR is None:
    raise FileNotFoundError("train.py not found - add the unet_project dataset with Add Input.")
os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
if not os.path.isdir(DATA_ROOT):
    raise FileNotFoundError(f"{DATA_ROOT} not found - add the images/labels dataset with Add Input.")

try:
    import rasterio  # noqa
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rasterio"], check=True)

from dataset import LandCoverDataset, CLASS_NAMES, NUM_CLASSES, IGNORE_INDEX

# run name = the same rule train.py uses
if MODEL in ("unet", "attn_unet"):
    RUN_NAME = f"{MODEL}_w32_l4_s{SEED}"
else:
    RUN_NAME = f"{MODEL}_{ENCODER}_{'pre' if PRETRAINED else 'scratch'}_s{SEED}"
RUN_DIR = os.path.join(OUT_ROOT, RUN_NAME)
FIG_DIR = os.path.join(RUN_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

# colours: built-up red, water blue, trees green, barren sand, unlabeled/padding white
PALETTE = np.array([[230, 60, 60], [30, 90, 255], [20, 160, 40], [230, 180, 60]], np.uint8)
IGNORE_COLOR = np.array([255, 255, 255], np.uint8)
LEGEND = [Patch(color=PALETTE[k] / 255, label=CLASS_NAMES[k]) for k in range(NUM_CLASSES)]

def colorize(lab):
    out = np.empty(lab.shape + (3,), np.uint8)
    out[:] = IGNORE_COLOR
    for k in range(NUM_CLASSES):
        out[lab == k] = PALETTE[k]
    return out

def save_fig(name):
    plt.savefig(os.path.join(FIG_DIR, name), dpi=150, bbox_inches="tight")

print("project :", PROJECT_DIR)
print("run name:", RUN_NAME)

## 1. The dataset with its labels

In [ ]:
splits = {s: LandCoverDataset(DATA_ROOT, s) for s in ["Train", "Validation", "Test"]}

rows = []
for s, ds in splits.items():
    c = ds.class_counts()
    rows.append({"split": s, "tiles": len(ds), **{n: round(100 * c[i] / c.sum(), 1) for i, n in enumerate(CLASS_NAMES)}})
share = pd.DataFrame(rows).set_index("split")
print("class share in % of labelled pixels")
display(share)

fig, ax = plt.subplots(figsize=(8, 3))
left = np.zeros(len(share))
for k, n in enumerate(CLASS_NAMES):
    ax.barh(share.index, share[n], left=left, color=PALETTE[k] / 255, label=n)
    left += share[n].values
ax.set_xlabel("% of labelled pixels")
ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, 1.25), frameon=False)
ax.invert_yaxis()
plt.tight_layout()
save_fig("class_share.png")
plt.show()

In [ ]:
def show_samples(ds, split, n=6):
    water = CLASS_NAMES.index("water")
    water_share = np.array([(lab == water).mean() for _, lab in ds.data])
    n_even = max(n - 2, 1)
    idx = list(np.unique(np.linspace(0, len(ds) - 1, n_even).round().astype(int)))
    idx += [int(i) for i in np.argsort(-water_share) if int(i) not in idx][: n - len(idx)]  # + tiles with most water
    fig, axes = plt.subplots(2, len(idx), figsize=(3 * len(idx), 6.6), squeeze=False)
    for j, i in enumerate(idx):
        img, lab = ds.data[i]
        axes[0, j].imshow(img[:3].transpose(1, 2, 0))
        axes[0, j].set_title(f"{split} tile {i}", fontsize=9)
        axes[1, j].imshow(colorize(lab))
        for r in (0, 1):
            axes[r, j].axis("off")
    axes[0, 0].text(-0.02, 0.5, "image", transform=axes[0, 0].transAxes, rotation=90, va="center", ha="right")
    axes[1, 0].text(-0.02, 0.5, "label", transform=axes[1, 0].transAxes, rotation=90, va="center", ha="right")
    fig.legend(handles=LEGEND, loc="lower center", ncol=NUM_CLASSES, frameon=False)
    fig.suptitle(f"{split}: image (top) and ground-truth label (bottom)", y=0.98)
    plt.tight_layout(rect=(0, 0.05, 1, 0.96))
    save_fig(f"samples_{split.lower()}.png")
    plt.show()

for s, ds in splits.items():
    show_samples(ds, s, N_SHOW)

## 2. Training
Calls `train.py`; the best checkpoint (by validation mIoU), `history.csv` and `metrics.json` are saved in the run folder.

In [ ]:
args = (f"--data_root {DATA_ROOT} --model {MODEL} --seed {SEED} --epochs {EPOCHS} "
        f"--batch_size {BATCH_SIZE} --out {OUT_ROOT} --name {RUN_NAME}")

if os.path.exists(os.path.join(RUN_DIR, "metrics.json")) and not RETRAIN:
    print("finished run found in", RUN_DIR, "- skipping training (set RETRAIN = True to train again)")
else:
    print("python train.py", args)
    !python train.py {args}

## 3. Training curves

In [ ]:
hist = pd.read_csv(os.path.join(RUN_DIR, "history.csv"))
res = json.load(open(os.path.join(RUN_DIR, "metrics.json")))
best_ep = res["best_epoch"]

fig, ax = plt.subplots(2, 2, figsize=(13, 8))
ax[0, 0].plot(hist["epoch"], hist["train_loss"], label="train")
ax[0, 0].plot(hist["epoch"], hist["val_loss"], label="validation")
ax[0, 0].set(title="Loss (cross-entropy + Dice)", xlabel="epoch", ylabel="loss")

ax[0, 1].plot(hist["epoch"], hist["val_mIoU"], color="tab:purple", label="val mIoU")
ax[0, 1].scatter([best_ep], [hist.loc[hist["epoch"] == best_ep, "val_mIoU"].iloc[0]], color="red", zorder=3, label=f"best (epoch {best_ep})")
ax[0, 1].set(title="Validation mIoU", xlabel="epoch", ylabel="mIoU")

for k, n in enumerate(CLASS_NAMES):
    ax[1, 0].plot(hist["epoch"], hist[f"val_IoU_{n}"], color=PALETTE[k] / 255, label=n)
ax[1, 0].set(title="Validation IoU per class", xlabel="epoch", ylabel="IoU")

ax[1, 1].plot(hist["epoch"], hist["lr"], color="tab:gray")
ax[1, 1].set(title="Learning rate (warm-up + cosine)", xlabel="epoch", ylabel="lr")

for a in ax.ravel():
    a.grid(alpha=0.3)
    if a.get_legend_handles_labels()[0]:
        a.legend()
    a.axvline(best_ep, color="red", ls=":", lw=1)
fig.suptitle(f"{RUN_NAME}: training curves", y=1.0)
plt.tight_layout()
save_fig("training_curves.png")
plt.show()
print(f"best epoch {best_ep} | {len(hist)} epochs trained | {hist['sec'].mean():.1f} s/epoch")

## 4. Confusion matrices

In [ ]:
def plot_cm(ax, mat, title):
    mat = np.array(mat, dtype=np.int64)               # rows = true class, columns = predicted class
    norm = mat / np.clip(mat.sum(1, keepdims=True), 1, None)
    ax.imshow(norm, vmin=0, vmax=1, cmap="Blues")
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, f"{100 * norm[i, j]:.1f}%\n({mat[i, j]:,})", ha="center", va="center", fontsize=8,
                    color="white" if norm[i, j] > 0.5 else "black")
    ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(CLASS_NAMES, rotation=20)
    ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(CLASS_NAMES)
    ax.set(title=title, xlabel="predicted", ylabel="true")

fig, ax = plt.subplots(1, 2, figsize=(15, 6))
plot_cm(ax[0], res["val"]["confusion"], f"Validation (best epoch {best_ep})")
plot_cm(ax[1], res["test"]["confusion"], f"Test (best epoch {best_ep})")
fig.suptitle("Confusion matrices, each row sums to 100% (diagonal = recall)", y=1.0)
plt.tight_layout()
save_fig("confusion_matrices.png")
plt.show()

## 5. Results on the test set

In [ ]:
def metrics_table(split):
    d = res[split]
    rows = [[c, p["iou"], p["precision"], p["recall"], p["f1"], p["support_px"]] for c, p in d["per_class"].items()]
    df = pd.DataFrame(rows, columns=["class", "IoU", "precision", "recall", "F1", "pixels"]).set_index("class")
    return df, d

for split in ("test", "val"):
    df, d = metrics_table(split)
    print(f"=== {split.upper()} ===   mIoU {d['mIoU']:.3f} | mF1 {d['mF1']:.3f} | overall accuracy {d['overall_acc']:.3f}")
    display(df.round(3))

# per-class IoU, validation vs test
iou = pd.DataFrame({s: [res[s]["per_class"][c]["iou"] for c in CLASS_NAMES] for s in ("val", "test")}, index=CLASS_NAMES)
ax = iou.plot.bar(figsize=(8, 4), rot=0, color=["#9ecae1", "#3182bd"])
ax.set(ylim=(0, 1), ylabel="IoU", title=f"{RUN_NAME}: IoU per class")
for cont in ax.containers:
    ax.bar_label(cont, fmt="%.2f", fontsize=8)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
save_fig("iou_per_class.png")
plt.show()

In [ ]:
import torch
from model import build_model

device = "cuda" if torch.cuda.is_available() else "cpu"
ck = torch.load(os.path.join(RUN_DIR, "best.pt"), map_location="cpu")
c = ck["args"]
net = build_model(c.get("model", "unet"), c.get("encoder", "resnet34"), False,
                  c.get("width", 32), c.get("levels", 4), c["dropout"], NUM_CLASSES)
net.load_state_dict(ck["model_state"])
net.to(device).eval()

test_ds = LandCoverDataset(DATA_ROOT, "Test")
test_ds.set_norm(ck["mean"], ck["std"])
mean = np.array(ck["mean"], np.float32)[:, None, None]
std = np.array(ck["std"], np.float32)[:, None, None]

idx = np.unique(np.linspace(0, len(test_ds) - 1, N_SHOW).round().astype(int))
CORRECT, WRONG = np.array([200, 230, 200], np.uint8), np.array([220, 30, 30], np.uint8)

fig, axes = plt.subplots(len(idx), 4, figsize=(12, 3 * len(idx)), squeeze=False)
with torch.no_grad():
    for r, i in enumerate(idx):
        x, y = test_ds[int(i)]
        pred = net(x.unsqueeze(0).to(device)).argmax(1)[0].cpu().numpy()
        lab = y.numpy()
        valid = lab != IGNORE_INDEX
        rgb = (np.clip(x.numpy() * std + mean, 0, 1) * 255).astype(np.uint8).transpose(1, 2, 0)
        rgb[~valid] = 255
        pred_vis = np.where(valid, pred, IGNORE_INDEX)
        err = np.empty(lab.shape + (3,), np.uint8)
        err[:] = IGNORE_COLOR
        err[valid & (pred == lab)] = CORRECT
        err[valid & (pred != lab)] = WRONG
        acc = (pred[valid] == lab[valid]).mean()
        for col, (im, t) in enumerate([(rgb, f"test tile {i}"), (colorize(lab), "ground truth"),
                                       (colorize(pred_vis), "prediction"), (err, f"errors (pixel acc {acc:.2f})")]):
            axes[r, col].imshow(im)
            axes[r, col].set_title(t, fontsize=9)
            axes[r, col].axis("off")
fig.legend(handles=LEGEND + [Patch(color=WRONG / 255, label="wrong pixel")], loc="lower center", ncol=NUM_CLASSES + 1, frameon=False)
plt.tight_layout(rect=(0, 0.03, 1, 1))
save_fig("test_predictions.png")
plt.show()

## 6. What the attention gates do

In [ ]:
# attention gates: what does each gate let through?  (bright = skip features kept, dark = suppressed)
!python attention_maps.py --checkpoint {RUN_DIR}/best.pt --data_root {DATA_ROOT} --split Test --n 6 --out {FIG_DIR}/attention
from IPython.display import Image
display(Image(os.path.join(FIG_DIR, "attention", "attention_test.png")))

## Summary

In [ ]:
t = res["test"]
print(f"{RUN_NAME}")
print(f"  best epoch : {best_ep}")
print(f"  val  mIoU  : {res['val']['mIoU']:.3f}")
print(f"  test mIoU  : {t['mIoU']:.3f} | mF1 {t['mF1']:.3f} | accuracy {t['overall_acc']:.3f}")
print(f"  figures    : {FIG_DIR}")
print()
print("To get the 3-seed average: set SEED = 2, then 3, re-run the notebook, then run:")
print(f"  !python summarize.py --runs {OUT_ROOT}")